# LLM Evaluation Metrics for Business Applications

**Course:** Natural Language Processing · Unit 6 — Advanced Applications and Evaluation  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 10 — Machine Translation and Encoder–Decoder Models  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Compute **F1-score** for NER evaluation in a document processing scenario
2. Evaluate text generation quality with **BLEU** and **ROUGE** metrics
3. Compare model output quality with **Perplexity** as an intrinsic LM metric
4. Use **BERTScore** to capture semantic similarity beyond n-gram overlap
5. Choose the right metric for each NLP business task using a decision guide

## Why Multiple Metrics?

No single metric captures all dimensions of text quality:

| Metric | What it captures | What it misses |
|---|---|---|
| F1-score | Precision/recall balance | Ordering, fluency |
| BLEU | N-gram precision vs reference | Semantics, paraphrase |
| ROUGE | N-gram recall vs reference | Fluency, coherence |
| Perplexity | Language model confidence | Task performance |
| BERTScore | Semantic similarity | Factual accuracy |


---

## 1. Environment Setup

Install dependencies **once** in your terminal:

```bash
pip install evaluate sacrebleu rouge-score bert-score transformers torch
```


In [ ]:
# pip install evaluate sacrebleu rouge-score bert-score transformers torch  # run once
import math
import torch
import evaluate
from transformers import GPT2LMHeadModel, GPT2Tokenizer


---

## 2. Business Case 1 — F1-Score for NER in Invoice Processing

**Scenario:** An accounts-payable team uses an NER system to extract `AMOUNT`, `VENDOR`, and `DATE` entities from invoices. We evaluate the system's precision and recall on a sample of 10 invoices.


In [ ]:
# Ground-truth (gold) entity spans for 10 invoices
gold_entities = [
    {'AMOUNT', 'VENDOR', 'DATE'},
    {'AMOUNT', 'DATE'},
    {'VENDOR', 'DATE'},
    {'AMOUNT', 'VENDOR'},
    {'AMOUNT', 'VENDOR', 'DATE'},
    {'DATE'},
    {'AMOUNT', 'VENDOR', 'DATE'},
    {'AMOUNT'},
    {'VENDOR', 'DATE'},
    {'AMOUNT', 'VENDOR', 'DATE'},
]

# Predicted entity spans from the NER system
pred_entities = [
    {'AMOUNT', 'VENDOR'},          # missed DATE
    {'AMOUNT', 'DATE'},            # correct
    {'VENDOR'},                    # missed DATE
    {'AMOUNT', 'VENDOR', 'DATE'},  # extra DATE (false positive)
    {'AMOUNT', 'VENDOR', 'DATE'},  # correct
    {'DATE'},                      # correct
    {'AMOUNT', 'DATE'},            # missed VENDOR
    {'AMOUNT'},                    # correct
    {'VENDOR', 'DATE'},            # correct
    {'AMOUNT', 'VENDOR', 'DATE'},  # correct
]


def entity_f1(gold_list: list, pred_list: list) -> tuple:
    """Compute micro-averaged precision, recall, and F1 for entity sets.

    Args:
        gold_list: List of sets of gold entity labels per document.
        pred_list: List of sets of predicted entity labels per document.

    Returns:
        Tuple of (precision, recall, f1) as floats in [0, 1].
    """
    tp = sum(len(g & p) for g, p in zip(gold_list, pred_list))
    fp = sum(len(p - g) for g, p in zip(gold_list, pred_list))
    fn = sum(len(g - p) for g, p in zip(gold_list, pred_list))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1


precision, recall, f1 = entity_f1(gold_entities, pred_entities)
print(f'NER Evaluation on Invoice Dataset')
print(f'  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1-score:  {f1:.4f}')


> **Output interpretation:** Micro-averaged F1 counts individual entity instances across all documents. A precision below recall means the system misses entities (false negatives > false positives). For an invoice-processing system, **recall** is typically more critical — missing an AMOUNT entity costs more than occasionally extracting a spurious one.


### 2.1 Manual F1 Calculation — Understanding the Formula


In [ ]:
# Manually show TP / FP / FN counts
tp_total = sum(len(g & p) for g, p in zip(gold_entities, pred_entities))
fp_total = sum(len(p - g) for g, p in zip(gold_entities, pred_entities))
fn_total = sum(len(g - p) for g, p in zip(gold_entities, pred_entities))

print(f'True Positives  (TP): {tp_total}  — correctly predicted entities')
print(f'False Positives (FP): {fp_total}  — predicted but not in gold')
print(f'False Negatives (FN): {fn_total}  — in gold but not predicted')
print()
print(f'Precision = TP / (TP + FP) = {tp_total} / {tp_total + fp_total} = {tp_total / (tp_total + fp_total):.4f}')
print(f'Recall    = TP / (TP + FN) = {tp_total} / {tp_total + fn_total} = {tp_total / (tp_total + fn_total):.4f}')


> **Output interpretation:** Decomposing into TP/FP/FN makes errors transparent. A high FP count suggests the model is over-predicting (hallucinating entities). A high FN count means it is under-predicting (missing entities). Fixing FP usually requires increasing the classification confidence threshold; fixing FN requires more training data or a lower threshold.


---

## 3. Business Case 2 — BLEU and ROUGE for Report Summarisation

**Scenario:** Two generative models (T5 and GPT-2) produce summaries of a quarterly earnings report. We use BLEU and ROUGE to compare their quality against a human-written reference.


In [ ]:
bleu = evaluate.load('sacrebleu')
rouge = evaluate.load('rouge')

reference = (
    'Bancolombia reported a 15% increase in net income for Q3 2024, '
    'reaching 2.3 trillion pesos. Digital channels handled 78% of transactions.'
)

system_summaries = {
    'T5-small': (
        'bancolombia reported net income of 2.3 trillion pesos '
        'in the third quarter, up 15 percent from last year.'
    ),
    'GPT-2': (
        'the bank had strong earnings growth and digital adoption '
        'continued to accelerate in the colombian financial sector.'
    ),
}

for model_name, summary in system_summaries.items():
    b = bleu.compute(predictions=[summary], references=[[reference]])
    r = rouge.compute(predictions=[summary], references=[reference])
    print(f'[{model_name}]')
    print(f'  Summary: {summary}')
    print(f'  BLEU:    {b["score"]:.2f}')
    print(f'  ROUGE-1: {r["rouge1"]:.4f}')
    print(f'  ROUGE-L: {r["rougeL"]:.4f}')
    print()


> **Output interpretation:** T5 typically scores higher on BLEU because it more closely reproduces the reference phrasing. GPT-2 uses more paraphrase ('strong earnings growth' vs '15% increase') which BLEU penalises despite conveying similar meaning. This illustrates BLEU's core limitation: it is a surface-form metric, not a semantic one. ROUGE-L is slightly more forgiving of paraphrase because it looks for the longest common subsequence rather than exact n-gram matches.


---

## 4. Business Case 3 — Perplexity for Text Quality Assessment

**Scenario:** A content team uses a language model to detect whether machine-generated text is fluent enough for publication. Perplexity provides an automatic fluency score.

**Formula:** PPL(text) = exp( − (1/N) Σ log P(token_i | context) )

Lower perplexity = the model assigns higher probability to the text = more fluent/expected text.


In [ ]:
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model.eval()


def compute_perplexity(model: GPT2LMHeadModel, tokenizer: GPT2Tokenizer, text: str) -> float:
    """Compute GPT-2 perplexity for a given text string.

    Args:
        model: Pre-trained GPT-2 model in evaluation mode.
        tokenizer: Corresponding GPT-2 tokeniser.
        text: Input text to evaluate.

    Returns:
        Perplexity score (float, lower is better).
    """
    encodings = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**encodings, labels=encodings['input_ids'])
    return math.exp(outputs.loss.item())


texts = {
    'Fluent (human)': 'The quarterly earnings report shows strong revenue growth across all business segments.',
    'Disfluent (noisy)': 'quarterly report shows earnings strong grow all segments across business.',
    'Formal Spanish in English model': 'El reporte trimestral muestra crecimiento en todos los segmentos.',
}

for label, text in texts.items():
    ppl = compute_perplexity(gpt2_model, gpt2_tokenizer, text)
    print(f'[{label}]')
    print(f'  Text: {text}')
    print(f'  Perplexity: {ppl:.2f}')
    print()


> **Output interpretation:** The human-written fluent text gets the lowest perplexity (GPT-2 assigns high probability to natural English). The word-order scrambled text has higher perplexity. The Spanish text has the highest perplexity because GPT-2 is an English model — it treats Spanish as highly unexpected input. This property makes perplexity useful for language detection and fluency filtering.


---

## 5. Business Case 4 — BERTScore: Semantic Evaluation

**Scenario:** Two customer service response systems produce different phrasings of the correct answer. BLEU penalises paraphrase, but BERTScore measures semantic similarity using contextual embeddings.


In [ ]:
# pip install bert-score  # run once if not already installed
from bert_score import score as bert_score_fn

reference_response = 'Your account balance is 500,000 pesos. Please visit the branch to update your information.'

system_responses = [
    'The current balance in your account is 500,000 COP. To update your data, please come to our office.',  # paraphrase
    'I am sorry, I cannot find your account. Please try again later.',  # wrong answer
]

references = [reference_response, reference_response]

P, R, F1 = bert_score_fn(system_responses, references, lang='en', verbose=False)
print('BERTScore results (precision / recall / F1):')
labels = ['Paraphrase (correct)', 'Wrong answer']
for label, p, r, f in zip(labels, P, R, F1):
    print(f'  [{label}] P={p:.4f}  R={r:.4f}  F1={f:.4f}')


> **Output interpretation:** BERTScore uses contextual token embeddings (from RoBERTa by default) to match tokens between prediction and reference by semantic similarity rather than exact string match. The correct paraphrase should score near 0.9 F1; the wrong answer should score much lower. This is BERTScore's key advantage over BLEU/ROUGE for customer-facing applications where paraphrase quality matters.


---

## 6. Metric Selection Guide


In [ ]:
import pandas as pd

guide = pd.DataFrame([
    {'Task': 'Named Entity Recognition', 'Primary metric': 'F1-score (micro/macro)', 'Why': 'Balances precision and recall for span-level labels'},
    {'Task': 'Machine Translation', 'Primary metric': 'BLEU', 'Why': 'Standard benchmark metric for MT leaderboards'},
    {'Task': 'Text Summarisation', 'Primary metric': 'ROUGE-1 / ROUGE-L', 'Why': 'Measures content coverage vs reference summary'},
    {'Task': 'Language Fluency Filter', 'Primary metric': 'Perplexity', 'Why': 'Intrinsic measure of how natural the text is'},
    {'Task': 'Customer Service QA', 'Primary metric': 'BERTScore F1', 'Why': 'Captures semantic equivalence, handles paraphrase'},
    {'Task': 'Document Classification', 'Primary metric': 'Accuracy + Macro F1', 'Why': 'Accuracy when balanced; macro F1 when imbalanced'},
])
print(guide.to_string(index=False))


> **Output interpretation:** This guide summarises when to use each metric. In practice, always report **multiple metrics** to give a complete picture. For a new business NLP project, start with the primary metric from this table, then add BERTScore and human evaluation for the most critical outputs.


---

## Summary

| Metric | Library | When to use |
|---|---|---|
| F1-score | `sklearn.metrics` | Classification, NER, information retrieval |
| BLEU | `evaluate` (sacrebleu) | MT, seq2seq with fixed reference |
| ROUGE | `evaluate` (rouge-score) | Summarisation, abstractive generation |
| Perplexity | `transformers` | Language model quality, fluency filtering |
| BERTScore | `bert-score` | Any task where paraphrase is acceptable |
